In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/val.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/train.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/test.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/dataset.csv


In [2]:
!pip install transformers datasets torch scikit-learn underthesea -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.2 MB/s eta 0:00:00


In [3]:
import pandas as pd

BASE = "/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0"
train_df = pd.read_csv(f"{BASE}/train.csv")
val_df   = pd.read_csv(f"{BASE}/val.csv")
test_df  = pd.read_csv(f"{BASE}/test.csv")
print(train_df.shape, val_df.shape, test_df.shape)

(2993, 9) (374, 9) (375, 9)


In [4]:
from underthesea import word_tokenize
from tqdm import tqdm
tqdm.pandas()

def segment(text):
    return word_tokenize(str(text), format="text")

train_df["text_seg"] = train_df["text"].progress_apply(segment)
val_df["text_seg"]   = val_df["text"].progress_apply(segment)
test_df["text_seg"]  = test_df["text"].progress_apply(segment)
print("Done!")

100%|██████████| 375/375 [00:08<00:00, 42.86it/s]

Done!


In [5]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class ViDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(df["text_seg"]),
            truncation=True,
            padding=True,
            max_length=max_len
        )
        self.labels = list(df["label_int"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_ds = ViDataset(train_df, tokenizer)
val_ds   = ViDataset(val_df,   tokenizer)
test_ds  = ViDataset(test_df,  tokenizer)
print("Datasets ready:", len(train_ds), len(val_ds), len(test_ds))

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Datasets ready: 2993 374 375


In [6]:
from transformers import (AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:, 1]
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="macro"),
        "auroc":    roc_auc_score(labels, probs)
    }

args = TrainingArguments(
    output_dir="./phobert-ai-detection",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.05,        # ← tăng từ 0.01 lên 0.05
    learning_rate=2e-5,
    label_smoothing_factor=0.1, # ← giảm overconfident
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Auroc
1,1.321949,0.648059,0.919786,0.919267,0.993623
2,0.507382,0.433987,0.989305,0.989304,0.999800
3,0.445771,0.476847,0.983957,0.983953,0.999657
4,0.414880,0.571802,0.959893,0.959828,0.999685


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=376, training_loss=0.5887338785415, metrics={'train_runtime': 367.2509, 'train_samples_per_second': 40.749, 'train_steps_per_second': 1.28, 'total_flos': 1574982777384960.0, 'train_loss': 0.5887338785415, 'epoch': 4.0})

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(test_ds)
preds  = np.argmax(pred_output.predictions, axis=-1)
probs  = softmax(pred_output.predictions, axis=-1)[:, 1]
labels = pred_output.label_ids

print("=== Test Metrics ===")
print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"F1 (macro): {f1_score(labels, preds, average='macro'):.4f}")
print(f"AUROC    : {roc_auc_score(labels, probs):.4f}")
print("\n=== Classification Report ===")
print(classification_report(labels, preds, target_names=["human", "ai"]))
print("=== Confusion Matrix ===")
print(confusion_matrix(labels, preds))

=== Test Metrics ===
Accuracy : 0.9680
F1 (macro): 0.9680
AUROC    : 0.9984

=== Classification Report ===
              precision    recall  f1-score   support

       human       0.99      0.94      0.97       188
          ai       0.94      0.99      0.97       187

    accuracy                           0.97       375
   macro avg       0.97      0.97      0.97       375
weighted avg       0.97      0.97      0.97       375

=== Confusion Matrix ===
[[177  11]
 [  1 186]]


In [8]:
from scipy.special import softmax
import numpy as np
import matplotlib.pyplot as plt

# Lấy predictions trên test set
pred_output = trainer.predict(test_ds)
probs = softmax(pred_output.predictions, axis=-1)
max_probs = probs.max(axis=-1)  # confidence cao nhất mỗi mẫu

print("=== Confidence Distribution ===")
print(f"Mean confidence : {max_probs.mean():.4f}")
print(f"Min confidence  : {max_probs.min():.4f}")
print(f"Max confidence  : {max_probs.max():.4f}")
print(f"% > 0.99        : {(max_probs > 0.99).mean():.1%}")
print(f"% > 0.95        : {(max_probs > 0.95).mean():.1%}")
print(f"% < 0.80        : {(max_probs < 0.80).mean():.1%}")

=== Confidence Distribution ===
Mean confidence : 0.9508
Min confidence  : 0.5205
Max confidence  : 0.9630
% > 0.99        : 0.0%
% > 0.95        : 92.5%
% < 0.80        : 2.4%


In [9]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# T=1.0
probs_t1 = softmax(pred_output.predictions, axis=-1)[:, 1]

# T=2.0
probs_t2 = softmax(pred_output.predictions / 2.0, axis=-1)[:, 1]

labels = pred_output.label_ids

# ECE (Expected Calibration Error) - càng thấp càng tốt
def ece_score(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() > 0:
            acc = labels[mask].mean()
            conf = probs[mask].mean()
            ece += mask.sum() * abs(acc - conf)
    return ece / len(labels)

print(f"ECE T=1.0: {ece_score(probs_t1, labels):.4f}")
print(f"ECE T=2.0: {ece_score(probs_t2, labels):.4f}")

ECE T=1.0: 0.0398
ECE T=2.0: 0.1646


In [10]:
import os

trainer.save_model("/kaggle/working/phobert-finetuned-v2")
tokenizer.save_pretrained("/kaggle/working/phobert-finetuned-v2")

print("Files saved:")
for f in os.listdir("/kaggle/working/phobert-finetuned-v2"):
    size = os.path.getsize(f"/kaggle/working/phobert-finetuned-v2/{f}")
    print(f"{f}: {size/1024/1024:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Files saved:
bpe.codes: 1.1 MB
config.json: 0.0 MB
tokenizer_config.json: 0.0 MB
model.safetensors: 515.0 MB
vocab.txt: 0.9 MB
added_tokens.json: 0.0 MB
training_args.bin: 0.0 MB
